In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
import time

# ============================
# CONFIG
# ============================
DATA_DIR = r"C:\Users\m8494\Downloads\7030data"
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
IMG_SIZE = 160
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-4
NUM_CLASSES = 3
MODEL_SAVE = 'efficientnet_b0_cpu.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.set_num_threads(4)

# ============================
# TRANSFORMS
# ============================
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop((IMG_SIZE, IMG_SIZE), scale=(0.8,1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# ============================
# DATALOADERS
# ============================
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_tfms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class_names = train_dataset.classes
print("Classes:", class_names)
print("Train images:", len(train_dataset), "Val images:", len(val_dataset))
print("Device:", DEVICE)

# ============================
# MODEL
# ============================
model = timm.create_model('efficientnet_b0', pretrained=True)
in_features = model.classifier.in_features if hasattr(model, 'classifier') else model.get_classifier().in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# ============================
# METRIC STORAGE
# ============================
train_loss_hist = []
val_loss_hist = []
train_acc_hist = []
val_acc_hist = []
prec_hist = []
recall_hist = []
f1_hist = []
epoch_time_hist = []

# ============================
# TRAINING LOOP
# ============================
best_val_acc = 0.0
total_start = time.time()

for epoch in range(1, EPOCHS+1):

    epoch_start = time.time()

    # ---------- TRAIN ----------
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False)
    for images, labels in loop:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)
        loop.set_postfix(loss=running_loss/running_total, acc=100*running_correct/running_total)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    # METRICS
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    epoch_time = time.time() - epoch_start

    # store metrics
    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)
    train_acc_hist.append(train_acc)
    val_acc_hist.append(val_acc)
    prec_hist.append(precision)
    recall_hist.append(recall)
    f1_hist.append(f1)
    epoch_time_hist.append(epoch_time)

    # PRINT SUMMARY
    print(f"Epoch {epoch}/{EPOCHS} -> "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%, "
          f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, "
          f"Time: {epoch_time:.2f}s")

    scheduler.step(val_acc)

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state': model.state_dict(), 'class_names': class_names}, MODEL_SAVE)
        print(f"Saved best model (val_acc={best_val_acc*100:.2f}%) -> {MODEL_SAVE}")

total_end = time.time()
total_time = total_end - total_start

# ============================
# AVERAGE METRICS
# ============================
print("\n========== AVERAGE METRICS ==========")
print(f"Avg Train Loss: {np.mean(train_loss_hist):.4f}")
print(f"Avg Val Loss: {np.mean(val_loss_hist):.4f}")
print(f"Avg Train Accuracy: {np.mean(train_acc_hist)*100:.2f}%")
print(f"Avg Val Accuracy: {np.mean(val_acc_hist)*100:.2f}%")
print(f"Avg Precision: {np.mean(prec_hist):.4f}")
print(f"Avg Recall: {np.mean(recall_hist):.4f}")
print(f"Avg F1-score: {np.mean(f1_hist):.4f}")
print(f"Avg Time/Epoch: {np.mean(epoch_time_hist):.2f}s")
print(f"Total Training Time: {total_time:.2f}s ({total_time/60:.2f} min)")
print("======================================")

print("\nTraining complete. Best validation accuracy:", best_val_acc)

Classes: ['grade1', 'grade2', 'grade3']
Train images: 420 Val images: 180
Device: cpu


Epoch 1/20 -> Train Loss: 1.0567, Train Acc: 40.00%, Val Loss: 0.9625, Val Acc: 57.78%, Precision: 0.5678, Recall: 0.5778, F1: 0.5616, Time: 179.30s
Saved best model (val_acc=57.78%) -> efficientnet_b0_cpu.pth


Epoch 2/20 -> Train Loss: 0.8728, Train Acc: 63.57%, Val Loss: 0.8320, Val Acc: 59.44%, Precision: 0.5711, Recall: 0.5944, F1: 0.5722, Time: 177.30s
Saved best model (val_acc=59.44%) -> efficientnet_b0_cpu.pth


Epoch 3/20 -> Train Loss: 0.7432, Train Acc: 67.38%, Val Loss: 0.7584, Val Acc: 65.00%, Precision: 0.6475, Recall: 0.6500, F1: 0.6487, Time: 174.17s
Saved best model (val_acc=65.00%) -> efficientnet_b0_cpu.pth


Epoch 4/20 -> Train Loss: 0.6153, Train Acc: 76.67%, Val Loss: 0.7471, Val Acc: 65.56%, Precision: 0.6464, Recall: 0.6556, F1: 0.6493, Time: 174.48s
Saved best model (val_acc=65.56%) -> efficientnet_b0_cpu.pth


Epoch 5/20 -> Train Loss: 0.5499, Train Acc: 78.10%, Val Loss: 0.7397, Val Acc: 64.44%, Precision: 0.6469, Recall: 0.6444, F1: 0.6456, Time: 174.59s


... (11 outputs left out)

Epoch 17/20 -> Train Loss: 0.1760, Train Acc: 93.10%, Val Loss: 1.1469, Val Acc: 66.67%, Precision: 0.6543, Recall: 0.6667, F1: 0.6578, Time: 178.21s


Epoch 18/20 -> Train Loss: 0.2192, Train Acc: 91.19%, Val Loss: 1.1438, Val Acc: 69.44%, Precision: 0.6935, Recall: 0.6944, F1: 0.6938, Time: 177.13s


Epoch 19/20 -> Train Loss: 0.2274, Train Acc: 89.52%, Val Loss: 1.1864, Val Acc: 67.78%, Precision: 0.6767, Recall: 0.6778, F1: 0.6764, Time: 173.72s


Epoch 20/20 -> Train Loss: 0.1932, Train Acc: 92.38%, Val Loss: 1.1638, Val Acc: 68.33%, Precision: 0.6736, Recall: 0.6833, F1: 0.6767, Time: 174.99s

========== AVERAGE METRICS ==========
Avg Train Loss: 0.4116
Avg Val Loss: 0.9766
Avg Train Accuracy: 82.64%
Avg Val Accuracy: 66.03%
Avg Precision: 0.6555
Avg Recall: 0.6603
Avg F1-score: 0.6557
Avg Time/Epoch: 174.93s
Total Training Time: 3498.97s (58.32 min)

Training complete. Best validation accuracy: 0.6944444444444444
